LLM Setup

In [32]:
from dotenv import load_dotenv
from pathlib import Path
import os
##from openai import OpenAI

root = Path.cwd()
for candidate in [root, root.parent, root.parent.parent]:
    env_path = candidate / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        break

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI API KEY is set")
else:
    raise ValueError("OPENAI_API_KEY is not set in environment variables")

OPENAI API KEY is set


In [33]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0.9, max_tokens=1000)
llm_openai.invoke("What is the Capital of Italy?").content

'The capital of Italy is Rome (Italian: Roma).'

In [34]:
from pydantic import BaseModel
from typing import Literal
class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

llm_structured_output.invoke("The movie was great!")

llm_schema(movie_summary_flag='positive')

In [35]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a movie review evaluator"),
    ("human","Please categories movie review as postive or negative:{input}")]) 

In [36]:
llm_structured_output = llm_openai.with_structured_output(llm_schema)

In [37]:
str_parser=StrOutputParser()

##Template needs inputs in the form of Dictionary ,Hence we need to convert Input to Dictionary

In [38]:
from langchain_core.runnables import RunnableLambda,RunnableParallel

def pydantic_json(input:llm_schema)-> str:
    return input.model_dump()["movie_summary_flag"]

pydantic_json_runnable=RunnableLambda(pydantic_json)    

### ***Parallel Chain 1

In [39]:
#Task 1

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","you are a post generator"),
    ("human","Crate a post for following text for linkedIn:{text}")
])

#Task 2

llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0.9, max_tokens=1000)

#Task 3
str_parser = StrOutputParser()

chain_linkedin=linkedin_prompt | llm_openai | str_parser



### ***Parallel Chain 2



In [40]:
def insta_chain(text:dict):

    text = text["text"] 

    #Task 1
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system","you are a post generator"),
        ("human","Crate a post for following text for Instagram:{text}")
    ])
    
    #Task 2
    
    llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0.9, max_tokens=1000)
    
    #Task 3
     
    str_parser = StrOutputParser()
     
    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)
    

##Final Orchestration

In [41]:
from langchain_core.runnables import RunnableBranch

In [42]:
condition_chain=RunnableBranch(
    (lambda x:"positive" in x, chain_linkedin),
    insta_chain_runnable
)

final_chain=prompt_template | llm_structured_output | pydantic_json_runnable | condition_chain

In [43]:
final_chain.invoke("KGF")

'Here are three LinkedIn-ready, positive posts you can use — pick the tone that fits you best or tell me which to tailor further.\n\n1) Short & punchy\nPositivity is a choice — and it changes everything. Choosing optimism fuels better decisions, stronger connections, and more resilience. Today I’m focusing on what I can control, celebrating small wins, and lifting others along the way. What made you smile at work this week? #Positivity #GrowthMindset #Gratitude\n\n2) Professional & reflective\nA positive mindset isn’t about ignoring challenges — it’s about how we respond to them. By focusing on solutions, celebrating progress (even small steps), and supporting teammates, we turn obstacles into opportunities. I’m grateful for the colleagues who bring energy and constructive optimism every day. How do you keep positivity alive in your team? #Leadership #TeamCulture #Positivity\n\n3) Team-focused & uplifting\nWhen we lead with positivity, we create momentum. Small acts — a timely thank-yo